# Step 3: Model Development (revision 1)

## Changes from the initial submission (reviewer response)
| Item | Initial submission | Revision 1 | Comments |
|---|---|---|---|
| Hyperparameters | Fixed values, no search | Randomized search with patient-level `StratifiedGroupKFold` CV **inside the training set** | R1-M3(a), R2-M5 |
| Early stopping | Used the **test set** as `eval_set` (information leak) | Removed; `n_estimators` is tuned as a hyperparameter | R1-M3(b), R2-M5 |
| Pre-processing | Imputation/scaling fitted on the training set outside CV | `Pipeline` (median imputation → scaling → model), fitted within each training fold | R2-M5 |
| Class imbalance | `scale_pos_weight = sqrt(neg/pos)` for XGB/LGB only | The same weight for all three models (RF: `class_weight={0:1, 1:w}`) | R1-M3(c) |
| Thresholds | `predict()` default 0.5 | Fixed 0.10 / 0.20 (guideline FN-risk thresholds), plus Youden / 80%-sensitivity thresholds from **training OOF** predictions | R1-M3(c), R2-M5 |
| Calibration | — | Logistic recalibration fitted on training OOF predictions (`*_recal`) | R1-M6, R2-M4 |
| Validation designs | Patient-level random 80:20 | + Temporal validation (train < 2023-01-01 ≤ test) | R2-m6, R1-M7 |

## Inputs and outputs
- **Input**: `../data/step2_processing_outputs/step2_all_tagt_feat`
- **Output**: `../data/step3_model_outputs/<experiment>/` (models, best params, CV results, OOF / test predictions, thresholds, split)


In [ ]:
import sys, os, platform, json, warnings
sys.path.insert(0, os.path.abspath("../src"))   # revision-1 shared modules (src/fnpred)
import numpy as np, pandas as pd, sklearn, lightgbm, xgboost
print("python", platform.python_version(), "| pandas", pd.__version__, "| numpy", np.__version__,
      "| scikit-learn", sklearn.__version__, "| lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__)

In [ ]:
from fnpred.modeling import TrainConfig, train_all, save_result, SEARCH_SPACES

DATA_DIR   = "../data"
INPUT_PATH = f"{DATA_DIR}/step2_processing_outputs/step2_all_tagt_feat"
OUTPUT_DIR = f"{DATA_DIR}/step3_model_outputs"

# Experiments. "main" reproduces the original patient-level 80:20 split (seed 42).
EXPERIMENTS = {
    "main":     TrainConfig(split_mode="group"),
    "temporal": TrainConfig(split_mode="temporal", temporal_cutoff="2023-01-01"),
    # Sensitivity analysis without therapeutic G-CSF in the outcome (R2-M2).
    # Requires the outcome-component columns produced by the revised step 2.
    "no_gcsf":  TrainConfig(split_mode="group", target="Event_no_gcsf"),
}
# 64-core environment: 8 parallel CV fits x 8 threads per model (edit if needed)
for cfg in EXPERIMENTS.values():
    cfg.n_jobs_search, cfg.n_jobs_model = 8, 8
SEARCH_SPACES

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(df.shape)
df.filter(regex="^Event").mean()

In [ ]:
results = {}
for name, cfg in EXPERIMENTS.items():
    if cfg.target not in df.columns:
        print(f"skip {name}: column '{cfg.target}' not found (run the revised step 2)")
        continue
    print(f"===== {name} =====")
    results[name] = train_all(df, cfg)
    save_result(results[name], cfg, f"{OUTPUT_DIR}/{name}")

In [ ]:
# Supplementary Table: search space and selected hyperparameters
for name, r in results.items():
    print(name); display(r.best_params)
    print(json.dumps(r.thresholds, indent=1))